# FlashNystrom vs the sub-quadratic field: MQAR recall (LR-swept)

Apples-to-apples multi-query associative recall. Every mixer runs in ONE
environment, same backbone / seeds / data, only the attention-slot operator
changes. Backends: `sdpa`, `linear_attention`, `nystrom_reference`,
`flash_nystrom`, `flash_nystrom_tc`, `hyena`, `mamba`.

**Fairness: per-method learning-rate sweep.** Different architectures have very
different LR sensitivities, so a single fixed LR biases the comparison toward
whichever method it suits. Following the Zoology (Arora et al. 2023) convention,
we sweep the base LR per method over `LR_GRID` and report the best. The range spans 5e-4 to 1e-1: 5e-4 is the LR Hyena's paper states for its synthetics (Table A.1), and the top end extends past 3e-2 where the Nystrom variants previously peaked. A grid should bracket every method's optimum, not end on it. (Hyena's own
implicit-filter LRs, 1e-3 / 1e-5, are architectural and set internally regardless
of the base LR.)

**GPU requirement.** The `flash_nystrom` kernel and `mamba-ssm` need compute
capability >= 8.0 (A100/L4/L40S/H100). On the free **T4 (7.5)** flash_nystrom is
skipped and Mamba falls back to a ~1000x slower pure-PyTorch scan. Use an
**A100 or L4** runtime.

MQAR is bimodal, so a few seeds do not resolve the operators finely; the sweep
identifies the best LR per method and reports its seeds.

In [ ]:
import torch, subprocess, sys

def run_streaming(cmd):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end=""); sys.stdout.flush()
    p.wait()
    return p.returncode

print(subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout)
CC = torch.cuda.get_device_capability()
print("compute capability:", CC, "| flash_nystrom + mamba-ssm supported:", CC >= (8, 0))

In [ ]:
# Clone (with the CUTLASS submodule the kernel build needs), compile flash_nystrom
# for this arch, and install the real Mamba CUDA kernels.
import os, torch
%cd /content
!rm -rf FlashNystrom
!git clone --recursive -q https://github.com/athrva98/FlashNystrom.git
%cd /content/FlashNystrom
!pip -q install einops
CC = torch.cuda.get_device_capability()
if CC >= (8, 0):
    os.environ["TORCH_CUDA_ARCH_LIST"] = f"{CC[0]}.{CC[1]}"
    os.environ["FLASH_NYSTROM_LAX_BUILD"] = "1"  # tolerate 3rd-party header warnings
    !pip install -e . --no-build-isolation
    # Real Mamba kernels. Their setup.py imports torch, so pip build isolation
    # fails at 'getting requirements to build wheel'; --no-build-isolation uses the
    # torch already installed. causal-conv1d first (mamba-ssm depends on it).
    !pip -q install ninja packaging
    !pip install causal-conv1d --no-build-isolation
    !pip install mamba-ssm --no-build-isolation
    import flash_nystrom
    print("flash_nystrom built, version", flash_nystrom.__version__)
    from paper.mqar.baselines import _HAS_MAMBA_CUDA
    print("Mamba CUDA kernels available:", _HAS_MAMBA_CUDA)
else:
    print("sm < 8.0: skipping kernel builds; pure-PyTorch backends only (Mamba slow)")

In [ ]:
# LR sweep: each backend x each LR x seeds. Resumable (skips existing JSONs).
import os, torch
N_SEEDS = 3                          # per (backend, lr); raise for tighter estimates
LR_GRID = [5e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1]   # base learning rate swept per method
BACKENDS = ["sdpa", "linear_attention", "nystrom_reference",
            "flash_nystrom", "flash_nystrom_tc", "hyena", "mamba"]
if torch.cuda.get_device_capability() < (8, 0):
    BACKENDS = [b for b in BACKENDS if "flash_nystrom" not in b]
OUT = "runs/mqar_lrsweep"
os.makedirs(OUT, exist_ok=True)
total = len(BACKENDS) * len(LR_GRID) * N_SEEDS
print(f"{total} runs = {len(BACKENDS)} backends x {len(LR_GRID)} LRs x {N_SEEDS} seeds")
for b in BACKENDS:
    for lr in LR_GRID:
        for s in range(N_SEEDS):
            out = f"{OUT}/mqar_{b}_lr{lr:g}_seed{s}.json"
            if os.path.exists(out):
                print("skip", out); continue
            print(f"\n===== {b}  lr={lr:g}  seed {s} =====", flush=True)
            run_streaming(["python", "-u", "-m", "paper.mqar.train", "--backend", b,
                "--seed", str(s), "--lr", str(lr), "--kappa_star", "0", "--seq_len", "256",
                "--num_kv_pairs", "16", "--num_landmarks", "64", "--newton_iter", "6",
                "--batch_size", "256", "--epochs", "64",
                "--num_train", "20000", "--num_test", "2000", "--out_json", out])

In [ ]:
import json, glob, statistics as st
from collections import defaultdict
runs = defaultdict(list)  # (backend, lr) -> [recall over seeds]
for f in glob.glob("runs/mqar_lrsweep/*.json"):
    d = json.load(open(f))
    if "lr" not in d: continue
    runs[(d["backend"], d["lr"])].append(d["best_recall"])
order = ["sdpa", "linear_attention", "nystrom_reference",
         "flash_nystrom", "flash_nystrom_tc", "hyena", "mamba"]
lrs = sorted({lr for (_, lr) in runs})

print("BEST LR per method (fair comparison: sweep LR, report best):")
print(f"  {'backend':<20} {'best_lr':>8} {'recall (mean +/- sd)':>22}  seeds")
for b in order:
    cfgs = [(lr, v) for (bb, lr), v in runs.items() if bb == b]
    if not cfgs: continue
    lr_b, v_b = max(cfgs, key=lambda x: st.mean(x[1]))
    m = st.mean(v_b); sd = st.stdev(v_b) if len(v_b) > 1 else 0.0
    seeds = ['%.1f' % x for x in sorted(v_b)]
    print(f"  {b:<20} {lr_b:>8.0e} {m:>10.2f} +/- {sd:<7.2f} {seeds}")

print("\nFull LR sweep (mean recall over seeds):")
print(f"  {'backend':<20} " + "  ".join(f"{lr:>8.0e}" for lr in lrs))
for b in order:
    row = []
    for lr in lrs:
        v = runs.get((b, lr))
        row.append(f"{st.mean(v):8.2f}" if v else f"{'--':>8}")
    print(f"  {b:<20} " + "  ".join(row))

## Hyena d>=N correctness gate (LR-swept)

Zoology shows gated convolutions like Hyena solve MQAR only once model dimension
`d >= N`. This runs Hyena at `d = 256, 512` (>= N = 256) across the LR grid (1
seed each) and reports the best recall per dimension. High recall there confirms
the vendored Hyena is faithful (it genuinely fails at d<N, not because it is
broken). Note: d=512 is slow.

In [ ]:
import os, json, glob
OUT = "runs/mqar_gate"
os.makedirs(OUT, exist_ok=True)
LR_GRID = [5e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1]
for d in [256, 512]:
    for lr in LR_GRID:
        out = f"{OUT}/hyena_d{d}_lr{lr:g}_seed0.json"
        if os.path.exists(out): continue
        print(f"\n===== hyena dim {d}  lr={lr:g}  seed 0 =====", flush=True)
        run_streaming(["python", "-u", "-m", "paper.mqar.train", "--backend", "hyena",
            "--seed", "0", "--dim", str(d), "--lr", str(lr), "--kappa_star", "0",
            "--seq_len", "256", "--num_kv_pairs", "16",
            "--batch_size", "256", "--epochs", "64", "--num_train", "20000",
            "--num_test", "2000", "--out_json", out])
for d in [256, 512]:
    recs = [(json.load(open(f))["lr"], json.load(open(f))["best_recall"])
            for f in glob.glob(f"{OUT}/hyena_d{d}_lr*.json")]
    if recs:
        lr_b, r_b = max(recs, key=lambda x: x[1])
        print(f"hyena d={d}: best recall {r_b:.1f}% (at lr={lr_b:g})")